In [ ]:
import cv2
import numpy as np

def preprocess_for_ocr(img_path):
    # 1. Load image
    img = cv2.imread(img_path)
    
    # 2. Grayscale: Removes color noise. OCR models work best on 1-channel images.
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # 3. Thresholding (Binarization): Turns the image into pure Black & White.
    # cv2.THRESH_OTSU automatically calculates the best threshold value.
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # 4. Denoising: Removes "salt and pepper" noise (little dots).
    denoised = cv2.fastNlMeansDenoising(thresh, None, 10, 7, 21)
    
    return denoised

# --- Teacher's Note: Why do this? ---
# - Normalization: Ensures pixel values are consistent (0 to 255).
# - Deskewing: If the paper is tilted, the AI struggles. Correcting rotation is vital.
# - Scaling: If the image is tiny, we upscale it. PaddleOCR likes at least 300 DPI.

In [ ]:
from paddleocr import PaddleOCR

# 1. Initialize with V3/PaddleX compatible arguments.
# Note: 'cls' (text direction classification) is now handled automatically
# via 'use_textline_orientation=True' during init — no need to pass it at predict time.
ocr = PaddleOCR(
    use_textline_orientation=True,  # Replaces the old cls=True at call time
    lang='en'
)

# 2. FIX: Use predict() instead of the deprecated ocr().
# predict() does NOT accept 'cls' — orientation is controlled at init above.
results = ocr.predict("test.jpg")

# 3. FIX: The new predict() returns a list of result objects (one per image).
# Each result object exposes rec_texts, rec_scores, and dt_polys.
if results:
    for page_result in results:
        rec_texts  = page_result.get('rec_texts',  [])
        rec_scores = page_result.get('rec_scores', [])
        if not rec_texts:
            print("No text detected on this page.")
            continue
        for idx, (text, confidence) in enumerate(zip(rec_texts, rec_scores)):
            print(f"Line {idx}: {text} (Conf: {confidence:.2f})")
else:
    print("Failed to detect text. Check if test.jpg exists and is readable.")

In [ ]:
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional
from datetime import datetime, timezone

# --- SCHEMA 1: NoSQL (The Raw "Review Queue" Model) ---
class MongoDocument(BaseModel):
    filename: str
    raw_text: str
    avg_confidence: float
    # Modern Python standard for UTC time
    extracted_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
    status: str = "pending"

# --- SCHEMA 2: SQL (The "Clean Store" Model) ---
class InvoiceSQL(BaseModel):
    invoice_id: str = Field(..., min_length=1)
    date: str
    total_amount: float

    # Modern Pydantic V2 standard
    @field_validator('total_amount')
    @classmethod
    def must_be_positive(cls, v: float) -> float:
        if v <= 0:
            raise ValueError('Total amount must be greater than zero')
        return v

# Note: If you see a Pydantic @validator deprecation warning, it is coming from
# inside PaddleOCR's own dependencies — NOT from this code. This cell is already
# correct Pydantic V2 style and requires no changes.